# Modelado — predicción de supervivencia

Segundo cuaderno del taller. El primero (`02_exploracion.ipynb`) respondía
*"¿qué se asocia con sobrevivir?"*; este responde *"dado un pasajero nuevo,
¿qué probabilidad tiene de sobrevivir?"*.

Lee la capa **Silver** a través de `flows/05_ml_models.py`, nunca los datos
directamente. Para reproducirlo desde cero:

```
make ingest && make silver && make gold
```

y luego *Restart & Run All*.

In [ ]:
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Los modulos con nombre numerico se importan con importlib (seccion 3 de la guia)
ml = importlib.import_module("flows.05_ml_models")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

X, y = ml.load_dataset()          # lee Silver; X con columnas candidatas, y = Survived
print(X.shape, round(y.mean(), 4))

## Parte 10 — Formulación del problema

Antes de entrenar nada hay que fijar cuatro cosas: la **variable objetivo**, el
**momento de predicción**, la **métrica** y el **baseline** a superar.

El orden importa. Si la métrica se define *después* de entrenar, es fácil
elegir la que mejor le quede al modelo que ya se tiene.

El documento completo está en `reports/problema_ml.md`.

### Paso 39. Qué se predice y cómo se mide

In [ ]:
print(f"Pasajeros:             {len(X)}")
print(f"Variables candidatas:  {X.shape[1]}")
print(f"Tasa de supervivencia: {y.mean():.4f}")
print()
print("Variables que entran al modelo:")
print(f"  numericas:   {ml.NUM_FEATURES}")
print(f"  categoricas: {ml.CAT_FEATURES}")
print(f"  sospechosas: {ml.CABIN_NUM + ml.CABIN_CAT}   <- se evaluan en el Paso 61")
print()
print(f"Metrica principal: {ml.PRIMARY_METRIC}")

### Paso 40. Auditoría de variables y fuga de datos

| Columna | ¿Existe al embarcar? | ¿Riesgo de fuga? | Decisión |
|---|---|---|---|
| `Survived`, `Survivor` | No, es el objetivo | **Fuga directa** | Excluir |
| `PassengerId`, `Ticket`, `Name` | Sí | Identificadores: se memorizan, no generalizan | Excluir (el título entra vía `TTitulo`) |
| `NSex`, `NEmbarked`, `NCabin`, `NTitulo` | — | Duplican a sus categóricas e imponen un orden que no existe | Excluir; se usa *one-hot* |
| `Age` imputada | — | Calculada con el promedio de **todos** los pasajeros | Usar `AgeOriginal` |
| `Person` | Derivada de `Age` y `Sex` | Redundante con ambas | Excluir del modelo base |
| `Cabin`, `TCabin`, `HasCabin` | **Dudoso** | ¿Se registró la cabina *después* del naufragio? | Evaluar con y sin ella (Paso 61) |

**La fuga más sutil es la edad.** La `Age` de Silver se imputó con el promedio
de los 891 pasajeros, incluidos los que terminarán en el conjunto de prueba.
Entrenar con ella le daría al modelo información que no debería tener. Por eso
`load_dataset()` recupera `AgeOriginal`, con sus nulos, y la imputación se hace
dentro del Pipeline, que la calcula solo con datos de entrenamiento.

In [ ]:
# Verificacion: la columna Age que entra al modelo conserva los nulos reales
print(f"Nulos en la Age del modelo: {X['Age'].isna().sum()}   (deben ser 177)")
print()
silver = pd.read_parquet(ml.SILVER / ml.SILVER_NAME)
print("Columnas de Silver que NO entran al modelo:")
for c in sorted(set(silver.columns) - set(X.columns)):
    print(f"  {c}")

**La pregunta de debate del Paso 40.**

Si `HasCabin` mejora la métrica, ¿significa que tener cabina protegía?

Probablemente no. El Paso 5 mostró que la cabina falta en el 18.5 % de primera
clase y en el 97.6 % de tercera: la ausencia del dato refleja **cómo se
compilaron los manifiestos**, no una característica del pasajero. Y cabe la
posibilidad de que se registrara con más detalle para quienes sobrevivieron
para declararla.

Un modelo que explota un artefacto del registro puede tener buenas métricas y
ser inútil para estimar riesgo real. Por eso `TCabin` y `HasCabin` quedan
marcadas como sospechosas y se evalúan por separado en el Paso 61.

### Paso 41. División estratificada

El conjunto de prueba es una evaluación **final** y se abre **una sola vez**,
en la Parte 16. Todas las decisiones intermedias —qué modelo, qué
hiperparámetros, qué umbral, qué variables— se toman con validación cruzada
sobre el entrenamiento.

`stratify=y` mantiene la proporción de sobrevivientes en ambos conjuntos.
`random_state` fija la semilla para que la división sea reproducible.

In [ ]:
X_train, X_test, y_train, y_test = ml.split(X, y)

print(f"Entrenamiento: {len(X_train)} pasajeros, tasa {y_train.mean():.4f}")
print(f"Prueba:        {len(X_test)} pasajeros, tasa {y_test.mean():.4f}")
print(f"Completo:      {len(X)} pasajeros, tasa {y.mean():.4f}")
print()
print(f"Diferencia entre tasas: {abs(y_train.mean() - y_test.mean()):.4f}")

> ⚠️ **A partir de aquí, `X_test` y `y_test` no se tocan** hasta la Parte 16.
> Mirarlos antes invalidaría la evaluación final: cualquier decisión tomada
> viéndolos ajustaría el modelo a ese conjunto concreto, y sus métricas
> dejarían de estimar el rendimiento sobre datos nuevos.

### Paso 42. Baselines: la vara a superar

Un modelo complejo que no supera a una regla simple no aporta valor.

- **Baseline ingenuo:** predecir siempre "no sobrevive".
- **Regla de género:** predecir que sobrevive toda mujer. Es la regla que
  cualquiera deduciría del EDA en cinco minutos.

In [ ]:
ml.baseline_scores(X_train, y_train)

**Lo que muestran estos dos números.**

El baseline ingenuo acierta en torno al 62 % **sin aprender nada**, y su recall
es exactamente 0: no identifica ni un solo superviviente. Es la prueba de que
**la exactitud sola engaña** cuando las clases están desbalanceadas.

La regla de género ronda el 79 % de exactitud. **Esa es la vara real.** Un
modelo de machine learning que quede por debajo no justifica su complejidad.

Por eso la métrica principal es **ROC-AUC** y no la exactitud: no depende del
umbral de decisión y mide la capacidad de ordenar pasajeros por riesgo, que es
lo que una aseguradora necesita.

In [ ]:
# Para tenerlo a mano al comparar modelos en la Parte 13
baselines = ml.baseline_scores(X_train, y_train)
VARA = baselines.set_index("baseline").loc["regla de genero", "accuracy"]
print(f"Vara a superar (exactitud de la regla de genero): {VARA:.4f}")

## Parte 11 — Preprocesamiento reproducible con `Pipeline`

Un `Pipeline` encadena los pasos —imputar, codificar, modelar— en **un solo
objeto** con `fit` y `predict`. Al hacer `fit`, cada paso aprende solo de los
datos que recibe; al predecir, aplica lo aprendido.

Eso garantiza el mismo preprocesamiento en entrenamiento, validación y
producción, sin duplicar código y sin que una estadística del conjunto de
prueba se filtre al de entrenamiento.

### Paso 43. Demostrar la fuga de datos

El experimento: calcular el promedio de edad de las mujeres con **todos** los
datos y con **solo el entrenamiento**, y comparar.

In [ ]:
media_todos = X.loc[X["Sex"] == "female", "Age"].mean()
media_train = X_train.loc[X_train["Sex"] == "female", "Age"].mean()

print(f"Promedio de edad de mujeres, TODOS los datos: {media_todos:.4f}")
print(f"Promedio de edad de mujeres, solo TRAIN:      {media_train:.4f}")
print(f"Diferencia:                                   {abs(media_todos - media_train):.4f}")
print()
print(f"Mujeres en total: {(X['Sex'] == 'female').sum()}")
print(f"Mujeres en train: {(X_train['Sex'] == 'female').sum()}")
print(f"Mujeres en test:  {(X_test['Sex'] == 'female').sum()}")

**Por qué la diferencia es pequeña aquí, y por qué sigue siendo un error.**

La diferencia es de centésimas porque hay cientos de mujeres y el 80 % de
ellas ya está en el entrenamiento: añadir el 20 % restante casi no mueve el
promedio.

Pero el tamaño del error no es lo que lo hace incorrecto. Al usar el promedio
de todos, **información del conjunto de prueba viaja al entrenamiento**: el
modelo "sabe", indirectamente, algo sobre pasajeros que debería desconocer por
completo. La evaluación final deja de estimar el rendimiento sobre datos
nuevos y pasa a estimarlo sobre datos que el modelo ya vio de refilón.

Y en otros conjuntos de datos la diferencia **sí** es grande: con más
faltantes, con grupos pequeños, o con variables de alta varianza, una
estadística calculada sobre todo puede cambiar mucho el resultado. El método
debe ser correcto independientemente de que el error resulte pequeño en este
caso concreto.

### Paso 44. `GroupAgeImputer`

La imputación del Paso 22, ahora como transformador de scikit-learn. Esa
conversión es exactamente lo que elimina la fuga: `fit()` calcula los
promedios con los datos que recibe —dentro de la validación cruzada, solo los
de entrenamiento de ese pliegue— y `transform()` los aplica.

Si aparece un grupo que `fit` nunca vio, usa el promedio global en vez de
dejar un nulo.

In [ ]:
from sklearn.dummy import DummyClassifier

pipe = ml.build_pipeline(DummyClassifier())
pre = pipe[:-1].fit(X_train, y_train)      # solo el preprocesamiento, ajustado en TRAIN

print("Promedios de edad que aprendio el imputador, por (Sex, Pclass):")
print(pre["age_imputer"].group_means_.round(2).to_string())
print(f"\nPromedio global de respaldo: {pre['age_imputer'].global_mean_:.2f}")

### Paso 45-46. Verificar el preprocesamiento

El preprocesador se ajusta con `X_train` y se aplica a `X_test`. Tres
comprobaciones: que no queden nulos, cuántas columnas salen, y que las
categóricas no lleven un orden implícito.

In [ ]:
Z = pre.transform(X_test)

print(f"Entrada:  {X_test.shape[1]} columnas")
print(f"Salida:   {Z.shape[1]} columnas")
print(f"¿Quedan NaN tras el preprocesamiento? {np.isnan(Z).any()}")
print()
print("Nombres de las columnas generadas:")
for nombre in pre["prep"].get_feature_names_out():
    print(f"  {nombre}")

**Las dos cosas que confirma esta salida.**

**No quedan nulos.** Los 177 faltantes de edad los rellenó el `GroupAgeImputer`
con promedios aprendidos *solo* en entrenamiento. Si hubiera quedado algún NaN,
varios modelos fallarían al entrenar.

**`Sex` salió como `cat__Sex_female` y `cat__Sex_male`**, dos columnas de 0 y 1,
no como un único número. Eso es *one-hot*, y evita el problema que anotamos en
el Paso 24: una codificación numérica le diría al modelo que `Q` (2) es mayor
que `S` (0), un orden que no existe.

`min_frequency=10` agrupa las categorías muy raras. La cubierta `T`, con un
solo pasajero, no genera una columna propia que el modelo podría memorizar.

### La decisión a justificar: ¿agrupar por `Sex` o por `Sex` y `Pclass`?

El Paso 22 imputaba la edad con el promedio por género. El pipeline usa un
grupo más específico: género **y** clase.

La comparación formal con validación cruzada llega en el Paso 48. Por ahora,
cuánto difieren los valores que cada variante asignaría.

In [ ]:
por_sexo = ml.GroupAgeImputer(group_cols=("Sex",)).fit(X_train)
por_sexo_clase = ml.GroupAgeImputer(group_cols=("Sex", "Pclass")).fit(X_train)

print("Agrupando solo por Sex:")
print(por_sexo.group_means_.round(2).to_string())
print("\nAgrupando por Sex y Pclass:")
print(por_sexo_clase.group_means_.round(2).to_string())
print("\nRango de valores imputados:")
print(f"  solo Sex:        {por_sexo.group_means_.min():.2f} a {por_sexo.group_means_.max():.2f}")
print(f"  Sex y Pclass:    {por_sexo_clase.group_means_.min():.2f} a {por_sexo_clase.group_means_.max():.2f}")

La agrupación por `Sex` y `Pclass` produce un rango de valores más amplio,
porque recoge algo que el EDA ya mostraba: los pasajeros de primera clase eran
notablemente mayores que los de tercera (Paso 10). Imputar a una mujer de
tercera clase con el promedio de *todas* las mujeres le asignaría una edad
sesgada hacia arriba.

A cambio, los grupos son más pequeños y cada promedio se estima con menos
datos. El Paso 48 dirá si esa especificidad compensa.

## Parte 12 — Modelos candidatos y validación cruzada

Un modelo puede **memorizar** el entrenamiento y fallar con datos nuevos: eso
es sobreajuste. La validación cruzada de *k* pliegues divide el entrenamiento
en *k* partes, entrena con *k−1* y evalúa en la restante, rotando.

La **media** de los *k* puntajes estima el desempeño; la **desviación** indica
la estabilidad.

Con 712 pasajeros de entrenamiento la varianza es considerable. **Una
diferencia de medio punto entre dos modelos no es evidencia de que uno sea
mejor.**

### Paso 47. Los seis candidatos

In [ ]:
for nombre, modelo in ml.get_candidates().items():
    print(f"  {nombre:8s} {type(modelo).__name__}")

Una familia variada a propósito: un baseline que no aprende nada, un modelo
lineal interpretable, un árbol simple, dos ensambles y uno basado en
distancia. Todos con la semilla fija.

`dummy` está incluido deliberadamente: si un modelo no lo supera con holgura,
el problema está en los datos o en el preprocesamiento, no en el modelo.

### Paso 48. Validación cruzada

Cada candidato se evalúa dentro de **su propio pipeline completo**: la
imputación de edad, el escalado y el one-hot se recalculan en cada pliegue
usando solo los datos de entrenamiento de ese pliegue. Eso es lo que hace que
las métricas sean honestas.

In [ ]:
%%time
resultados, por_pliegue = ml.cross_validate_candidates(
    X_train, y_train, return_folds=True
)
resultados

### Paso 49. Leer los resultados

Tres lecturas, en este orden: desempeño, estabilidad y sobreajuste.

In [ ]:
cols = ["modelo", "roc_auc_mean", "roc_auc_std", "accuracy_mean", "f1_mean"]
tabla = resultados[cols].copy()
tabla["supera_la_vara"] = tabla["accuracy_mean"] > VARA
tabla.round(4)

In [ ]:
# Dispersión por pliegue de la métrica principal: la media sola esconde la variabilidad
auc = por_pliegue[por_pliegue["metrica"] == "roc_auc"]
orden = resultados.sort_values("roc_auc_mean", ascending=False)["modelo"]

sns.boxplot(data=auc, x="modelo", y="valor", order=orden)
sns.stripplot(data=auc, x="modelo", y="valor", order=orden, color="black", size=4)
plt.axhline(0.5, color="red", linestyle="--", linewidth=1, label="azar")
plt.title("ROC-AUC por pliegue")
plt.ylabel("ROC-AUC")
plt.xlabel("")
plt.legend()
plt.show()

In [ ]:
# Brecha entrenamiento - validacion: cuanto peor le va al modelo con datos que no vio
brecha = resultados[["modelo", "train_roc_auc_mean", "roc_auc_mean",
                     "brecha_roc_auc", "fit_time_mean"]].copy()
brecha = brecha.sort_values("brecha_roc_auc", ascending=False)
brecha.round(4)

**Cómo leer la brecha.**

Es el ROC-AUC en entrenamiento menos el de validación. Mide cuánto peor le va
al modelo con datos que no vio.

- **Brecha cercana a 0:** el modelo generaliza bien, o es demasiado simple
  para memorizar nada (el caso de `dummy`).
- **Brecha grande:** el modelo memorizó parte del entrenamiento. Suele crecer
  con la flexibilidad: un bosque de 300 árboles sin profundidad limitada puede
  ajustarse casi perfectamente a los datos que vio.

Un modelo con buena media y brecha pequeña es preferible a uno con media
ligeramente mejor y brecha grande: el segundo promete algo que no va a
cumplir con datos nuevos.

### La decisión pendiente del Paso 45: ¿`Sex` o `Sex` + `Pclass`?

Ahora sí se puede comparar con validación cruzada en lugar de con intuición.

In [ ]:
from sklearn.base import clone

def auc_con_imputador(group_cols):
    """Valida el mejor modelo lineal cambiando solo el agrupamiento del imputador."""
    pipe = ml.build_pipeline(ml.get_candidates()["logreg"])
    pipe.set_params(age_imputer__group_cols=group_cols)
    from sklearn.model_selection import StratifiedKFold, cross_val_score
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    s = cross_val_score(clone(pipe), X_train, y_train, cv=cv, scoring="roc_auc")
    return s.mean(), s.std()

for cols_grupo in [("Sex",), ("Sex", "Pclass")]:
    media, desv = auc_con_imputador(cols_grupo)
    print(f"  agrupando por {cols_grupo!s:20s} ROC-AUC = {media:.4f} (+/- {desv:.4f})")

### Paso 50. Elegir finalistas

El criterio **no** es "el primero de la tabla". Son cinco dimensiones:

1. **Desempeño medio** en la métrica principal.
2. **Estabilidad** — la desviación entre pliegues.
3. **Brecha entrenamiento-validación** — cuánto sobreajusta.
4. **Interpretabilidad** — ¿se puede explicar una predicción a la gerencia?
5. **Costo computacional** — `fit_time_mean`.

Un modelo ligeramente peor pero estable y explicable puede ser mejor elección
que el primero de la tabla.

> ⚠️ **Regla de oro: nunca se elige un modelo mirando el conjunto de prueba.**
> Todo lo anterior se decidió con validación cruzada sobre el entrenamiento.

La justificación escrita va en `reports/problema_ml.md`, sección "Selección de
modelos".

## Parte 13 — Evaluación en profundidad

Un solo número oculta **qué** errores comete el modelo. La Parte 12 dejó tres
finalistas y una tabla de medias; aquí se abre esa media para ver a quién
acierta y a quién no.

La herramienta es la **predicción fuera de muestra**: cada pasajero se predice
con un modelo entrenado sin él. Así se puede mirar matriz de confusión,
curvas, calibración y errores por subgrupo **sin tocar el conjunto de
prueba**, que sigue cerrado hasta la Parte 16.

> Una advertencia que conviene tener presente todo el tiempo: el vector de
> probabilidades mezcla los 5 modelos de los 5 pliegues. Diagnostica el
> **procedimiento**, no un modelo concreto.

### Paso 51. Predicciones fuera de muestra y matriz de confusión

In [ ]:
%%time
# Una probabilidad honesta por pasajero: ninguna la emitió un modelo que ya lo vio
proba_oof = ml.oof_proba_finalists(X_train, y_train)
proba_oof.describe().round(4)

In [ ]:
# Matriz de confusión del líder, con el umbral por convención
from sklearn.metrics import classification_report

LIDER = "hgb"
proba = proba_oof[LIDER].to_numpy()
pred = (proba >= 0.5).astype(int)

display(ml.confusion_table(y_train, pred))
print(classification_report(y_train, pred,
                            target_names=["no sobrevivió", "sobrevivió"], digits=3))

In [ ]:
# Qué significa cada celda para la aseguradora
vn, fp, fn, vp = ml.confusion_table(y_train, pred).to_numpy().ravel()
razon = f"{fp / fn:.2f}" if fn else "n/d"

print(f'''
Verdaderos negativos : {vn:3d}   no sobrevivió y el modelo lo previó
Falsos positivos     : {fp:3d}   EL ERROR CARO: póliza con prima insuficiente
Falsos negativos     : {fn:3d}   se cobra de más o se rechaza a un cliente viable
Verdaderos positivos : {vp:3d}   sobrevivió y el modelo lo previó

Por cada falso negativo el modelo comete {razon} falsos positivos.
La sección 2 de reports/problema_ml.md declaró que el falso positivo es el
error caro, así que esa razón es la cifra a vigilar al mover el umbral.
''')

**Cómo leer la matriz.** Las filas son lo que ocurrió; las columnas, lo que el
modelo dijo. La diagonal son los aciertos y las dos celdas restantes son los
dos errores, que **no cuestan lo mismo**:

- **Falso positivo** — el modelo da por superviviente a quien no lo fue. Para
  la aseguradora es una póliza mal tarifada: pérdida directa.
- **Falso negativo** — el modelo da por muerto a quien sobrevivió. Se cobra de
  más o se rechaza a un cliente viable: ingreso no percibido, no pérdida.

El `classification_report` descompone la exactitud por clase. Conviene mirar
el **recall de la clase "sobrevivió"**: es la proporción de supervivientes que
el modelo encuentra, y es donde la exactitud global engaña más.

### Paso 52. Curvas ROC y de precisión-recall

In [ ]:
# Los tres finalistas en el mismo eje, con sus probabilidades fuera de muestra
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

fig, (iz, de) = plt.subplots(1, 2, figsize=(13, 5))
for nombre in proba_oof.columns:
    RocCurveDisplay.from_predictions(y_train, proba_oof[nombre], name=nombre, ax=iz)
    PrecisionRecallDisplay.from_predictions(y_train, proba_oof[nombre], name=nombre, ax=de)

iz.plot([0, 1], [0, 1], "r--", lw=1, label="azar")
de.axhline(y_train.mean(), color="red", ls="--", lw=1,
           label=f"azar ({y_train.mean():.3f})")
iz.set_title("ROC — fuera de muestra")
de.set_title("Precisión-Recall — fuera de muestra")
iz.legend(loc="lower right")
de.legend(loc="lower left")
plt.tight_layout()
plt.show()

**¿Cuándo es más informativa la curva PR que la ROC?** Cuando la clase
positiva es minoritaria.

La ROC enfrenta el recall con la **tasa de falsos positivos**, cuyo
denominador son todos los negativos. Si los negativos abundan, añadir falsos
positivos mueve poco ese denominador y la curva se ve bien aunque la precisión
sea mala. La curva PR usa como denominador los **positivos predichos**, que
son pocos, y por eso castiga cada falso positivo.

Con 38 % de supervivientes el desbalance es moderado y las dos curvas cuentan
una historia parecida. Pero la línea de azar lo dice todo: en la ROC es la
diagonal, en la PR es una horizontal a la altura de la tasa base. **La curva
PR es la que corresponde al problema de la aseguradora**, porque su eje
vertical es exactamente la precisión que la sección 2 declaró prioritaria.

### Paso 53. Calibración y elección del umbral

`predict_proba` devuelve un puntaje, y `0.5` es una convención heredada, no una
decisión. Dos preguntas distintas:

1. **¿La probabilidad dice la verdad?** Un modelo está *calibrado* si, entre
   los pasajeros a los que asigna 0.7, sobrevivió cerca del 70 %.
2. **¿Dónde cortar?** Eso depende del costo de cada error, y se decide con
   datos de validación.

In [ ]:
# Calibración: lo que el modelo promete vs. lo que ocurrió
base = y_train.mean()
print(f"Brier de {LIDER}: {ml.brier(y_train, proba):.4f}    "
      f"(0 = perfecto; predecir siempre la tasa base da {base * (1 - base):.4f})")

cal = ml.calibration_table(y_train, proba, n_bins=8)
cal

In [ ]:
# El mismo contenido, en gráfico, con el n de cada tramo anotado
ok = cal[cal["n"] > 0]

plt.figure(figsize=(6.5, 6.5))
plt.plot([0, 1], [0, 1], "r--", lw=1, label="calibración perfecta")
plt.plot(ok["proba_media"], ok["tasa_real"], "o-", label=LIDER)
for _, f in ok.iterrows():
    plt.annotate(f"n={int(f['n'])}", (f["proba_media"], f["tasa_real"]),
                 textcoords="offset points", xytext=(7, -11), fontsize=8)
plt.xlabel("probabilidad que promete el modelo")
plt.ylabel("tasa real observada")
plt.title(f"Calibración de {LIDER} — fuera de muestra")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**Cómo leer este gráfico.** Cada punto es un tramo de probabilidad. Si el
modelo estuviera calibrado, todos caerían sobre la diagonal.

- Punto **por debajo** de la diagonal: el modelo promete más de lo que ocurre
  (optimista).
- Punto **por encima**: promete menos (pesimista).

El `n` anotado es imprescindible: un tramo con 3 pasajeros puede desviarse 30
puntos por puro azar. Solo los tramos con decenas de pasajeros sostienen una
lectura. Por eso la tabla se calcula a mano y no con `calibration_curve`, que
descarta los tramos vacíos y no devuelve el tamaño de cada uno.

**El Brier** es el error cuadrático medio de la probabilidad. La vara no es
0.25 sino `p·(1−p)` con la tasa base: eso es lo que obtiene quien responde
siempre "38.4 %" sin mirar al pasajero.

In [ ]:
# Tres umbrales, tres políticas distintas
from sklearn.metrics import f1_score, precision_score, recall_score

f1_max = ml.best_threshold(y_train, proba, criterio="f1")
piso80 = ml.threshold_for_precision(y_train, proba, min_precision=0.80)


def fila(criterio, u):
    p = (proba >= u).astype(int)
    return {"criterio": criterio, "umbral": round(float(u), 4),
            "precision": round(precision_score(y_train, p, zero_division=0), 4),
            "recall": round(recall_score(y_train, p, zero_division=0), 4),
            "f1": round(f1_score(y_train, p, zero_division=0), 4),
            "falsos_positivos": int(((p == 1) & (y_train == 0)).sum()),
            "falsos_negativos": int(((p == 0) & (y_train == 1)).sum())}


print("¿se alcanzó el piso de precisión de 0.80?", piso80["alcanzado"])
pd.DataFrame([fila("convención (0.5)", 0.5),
              fila("máximo F1", f1_max["umbral"]),
              fila("precisión >= 0.80", piso80["umbral"])])

In [ ]:
# Qué cuesta mover el umbral: las tres métricas a lo largo de todo el rango
barrido = ml.threshold_sweep(y_train, proba)

plt.figure(figsize=(9, 5))
for col in ["precision", "recall", "f1"]:
    plt.plot(barrido["umbral"], barrido[col], label=col)
plt.axvline(0.5, color="gray", ls=":", lw=1.5, label="convención 0.5")
plt.axvline(f1_max["umbral"], color="tab:green", ls="--", lw=1,
            label=f"máx F1 ({f1_max['umbral']:.3f})")
plt.axvline(piso80["umbral"], color="tab:blue", ls="--", lw=1,
            label=f"precisión >= 0.80 ({piso80['umbral']:.3f})")
plt.xlabel("umbral de decisión")
plt.ylabel("valor")
plt.title(f"Qué cuesta mover el umbral — {LIDER}")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**La decisión.** Las tres filas de la tabla son tres políticas, no tres
calidades de modelo: el modelo es el mismo y solo cambia dónde se corta.

- **Máximo F1** trata los dos errores como igual de caros. La sección 2
  declaró que no lo son, así que optimizar F1 contradice el criterio escrito.
- **Precisión ≥ 0.80** sube el umbral: menos falsos positivos (el error caro)
  a costa de muchos más falsos negativos. Es coherente con lo declarado, pero
  hay que mirar cuánto recall se sacrifica antes de aceptarlo.
- **0.5** no está justificado por nada salvo la costumbre.

El umbral elegido se registra aquí y se congela en la Parte 17 como
`THRESHOLD`. Anota la decisión y su razón en `reports/problema_ml.md`: elegir
el umbral después de ver el conjunto de prueba sería exactamente la trampa que
el protocolo de la sección 5 se comprometió a evitar.

### Paso 54. Análisis de errores por subgrupos

Una exactitud global promedia sobre grupos que el modelo trata de forma muy
distinta. Un modelo con 82 % de exactitud puede estar acertando el 95 % en un
grupo y el 55 % en otro.

In [ ]:
# Exactitud, precisión y recall dentro de cada subgrupo (peor primero)
con_tramo = X_train.copy()
tramo = pd.cut(con_tramo["Age"], bins=[0, 16, 30, 50, 100],
               labels=["<16", "16-30", "30-50", "50+"])
con_tramo["TramoEdad"] = tramo.astype(object).fillna("sin edad")

pd.concat([ml.subgroup_report(con_tramo, y_train, pred, by=v)
           for v in ["Sex", "Pclass", "TramoEdad"]], ignore_index=True)

In [ ]:
# Los 10 errores en los que el modelo estaba más seguro
COLS = ["Sex", "Pclass", "Age", "Fare", "Familia", "TTitulo", "TCabin",
        "real", "proba", "pred"]
ml.confident_errors(X_train, y_train, proba, n=10)[COLS]

**Qué buscar en estas dos tablas.**

En la primera, el `n` manda: un subgrupo con 20 pasajeros no sostiene una
conclusión por mala que se vea su exactitud. Lo que importa es un grupo
**grande** donde el modelo rinda mucho peor que en el resto.

En la segunda, un error con probabilidad 0.51 es ruido; uno con 0.95 significa
que el modelo tiene una creencia firme y equivocada. Si esos diez casos
comparten algo —el mismo título, la misma clase, familias grandes— eso señala
una **variable que falta**, no un modelo mal ajustado.

#### Reflexión ética

El modelo se apoya en `Sex` y `Pclass`, que son las variables que gobernaron
la evacuación de 1912. Predecir bien **lo que ocurrió** no implica que *deba*
ocurrir así, ni que esas variables sean admisibles para decidir quién recibe
un servicio hoy; en la mayoría de jurisdicciones, tarificar por sexo es
ilegal.

Hay además una asimetría en los errores por subgrupo que merece nombrarse: si
el modelo falla sistemáticamente más en un grupo, trasladarlo a producción
traslada también ese trato desigual. Anota en `reports/problema_ml.md` en qué
contextos usarías este modelo y en cuáles no.

### Paso 55. Curva de aprendizaje: ¿más datos o un modelo distinto?

La pregunta operativa es dónde invertir: en conseguir más datos o en cambiar
el modelo. La curva de aprendizaje la responde entrenando con fracciones
crecientes del conjunto y mirando cómo evolucionan las dos métricas.

In [ ]:
%%time
pipe_lider = ml.build_pipeline(ml.get_candidates()[LIDER])
curva = ml.learning_curve_table(pipe_lider, X_train, y_train)
curva

In [ ]:
# Las dos curvas con su banda de ±1 desviación entre pliegues
plt.figure(figsize=(8.5, 5))
for col, etiqueta, color in [("train", "entrenamiento", "tab:blue"),
                             ("val", "validación", "tab:orange")]:
    media, desv = curva[f"{col}_mean"], curva[f"{col}_std"]
    plt.plot(curva["n_entrenamiento"], media, "o-", color=color, label=etiqueta)
    plt.fill_between(curva["n_entrenamiento"], media - desv, media + desv,
                     alpha=.2, color=color)
plt.xlabel("pasajeros usados para entrenar")
plt.ylabel("ROC-AUC")
plt.title(f"Curva de aprendizaje — {LIDER}")
plt.legend()
plt.grid(alpha=.3)
plt.show()

**El diagnóstico tiene dos lecturas opuestas.**

| Lo que se ve | Nombre | Qué hacer |
|---|---|---|
| Las dos curvas **convergen en un valor bajo** | sesgo alto | El modelo es demasiado simple o faltan variables. Más datos no arreglan nada. |
| Queda una **brecha grande** que se estrecha al crecer `n` | varianza alta | Más datos o más regularización sí ayudan. |
| La curva de validación **se aplana** antes del final | saturación | Duplicar los datos daría muy poco. El margen está en el modelo o en las variables. |

Mira la columna `brecha` de la tabla: si apenas se mueve entre el primer y el
último tamaño, la curva ya saturó y el camino es el ajuste de hiperparámetros
(Parte 14) y la revisión de variables (Parte 15), no más pasajeros.

### Cierre de la Parte 13

Lo que queda registrado para las partes siguientes:

1. **El umbral elegido** y su justificación → `reports/problema_ml.md`, y en la
   Parte 17 como `THRESHOLD`.
2. **El desempeño por subgrupos** → entra en la ficha del modelo
   (`reports/model_card.md`, Paso 69).
3. **El diagnóstico de la curva de aprendizaje** → decide cuánto esperar del
   ajuste de hiperparámetros de la Parte 14.

El conjunto de prueba sigue sin abrirse.

## Parte 14 — Ajuste de hiperparámetros

Los hiperparámetros son los diales que **no se aprenden de los datos**:
profundidad del árbol, regularización, tasa de aprendizaje. Se buscan dentro
de la validación cruzada del entrenamiento.

El Paso 55 dejó un diagnóstico preciso: **varianza alta**. `hgb` y `forest`
alcanzan ROC-AUC de 0.99 en entrenamiento desde los 113 pasajeros. La
búsqueda, por tanto, no va a buscar más capacidad sino **menos**: limitar
profundidad y hojas, exigir más observaciones por hoja, subir la penalización.

> **El riesgo que hay que medir.** Cuantas más combinaciones se prueban, más
> probable es que alguna gane **por azar**. El máximo de muchos puntajes
> ruidosos está sesgado hacia arriba aunque ninguna configuración sea
> realmente mejor. Por eso el Paso 58 no pregunta "¿cuál ganó?" sino
> "¿ganó algo de verdad?".

### Paso 56. Definir el espacio de búsqueda

In [ ]:
# Cuánto cuesta cada rejilla antes de lanzarla
rejillas = ml.get_param_grids()

filas = []
for nombre, rejilla in rejillas.items():
    combinaciones = 1
    for valores in rejilla.values():
        combinaciones *= len(valores)
    filas.append({"modelo": nombre,
                  "hiperparámetros": len(rejilla),
                  "combinaciones": combinaciones,
                  "ajustes": combinaciones * ml.settings.cv_folds})

display(pd.DataFrame(filas))

for nombre, rejilla in rejillas.items():
    print(f"\n{nombre}")
    for clave, valores in rejilla.items():
        print(f"  {clave:30s} {valores}")

**El prefijo `model__` no es decorativo.** Indica a qué paso del Pipeline
apunta cada dial. Como la búsqueda recorre el pipeline completo, el
preprocesamiento —imputación de edad, escalado, one-hot— **se reajusta en cada
pliegue de cada combinación**. Sin eso, la búsqueda tendría exactamente la
fuga que el Paso 43 demostró, multiplicada por el número de combinaciones.

Una nota sobre la rejilla de `logreg`: el enunciado propone además
`'model__penalty': ['l2']`. Se omite a propósito, porque scikit-learn 1.8
deprecó ese argumento y lo retira en la 1.10. Como la lista tenía un único
valor y `l2` ya es el comportamiento por defecto, **el espacio de búsqueda es
idéntico sin él** y la salida queda sin advertencias.

### Paso 57. Búsqueda con validación cruzada

`GridSearchCV` prueba todas las combinaciones sobre los mismos pliegues que
usó el Paso 48, y al terminar reajusta el mejor pipeline con todo el
entrenamiento.

In [ ]:
%%time
# Solo X_train: el conjunto de prueba no participa en ninguna decisión
busquedas = {n: ml.tune_model(X_train, y_train, name=n) for n in ml.FINALISTS}

In [ ]:
# Antes y después, con los mismos pliegues del Paso 48
por_defecto = resultados.set_index("modelo")["roc_auc_mean"]

pd.DataFrame([
    {"modelo": n,
     "por_defecto": por_defecto[n],
     "ajustado": round(b.best_score_, 4),
     "mejora": round(b.best_score_ - por_defecto[n], 4),
     "combinaciones": len(b.cv_results_["params"]),
     "mejores_params": b.best_params_}
    for n, b in busquedas.items()
]).sort_values("ajustado", ascending=False).reset_index(drop=True)

In [ ]:
# ¿Qué tan plana es la superficie? Las 10 mejores combinaciones del líder
ml.search_results(busquedas[LIDER], top=10)

**Lo que dice la columna `dif_vs_mejor`.** Si las diez primeras filas están
separadas por milésimas, la superficie es **plana**: el modelo es insensible a
esos diales en ese rango y la combinación ganadora lo es por ruido, no por
mérito. En ese caso lo correcto es quedarse con la **configuración más
simple** entre las equivalentes —menos profundidad, menos estimadores— porque
cuesta menos y generaliza igual.

Compara también `dif_vs_mejor` con `std_test_score`: si la diferencia entre la
primera y la décima es menor que la desviación entre pliegues de cualquiera de
ellas, no hay forma de distinguirlas con estos datos.

### Paso 58. ¿Mejoró de verdad?

`best_score_` es el máximo de muchos puntajes ruidosos, así que **está sesgado
hacia arriba**. Dos formas de comprobar si el ajuste aportó algo real.

In [ ]:
# Opción B — comparación pareada: los MISMOS pliegues para ambas versiones
pareado = ml.paired_fold_comparison(X_train, y_train, name=LIDER,
                                    best_params=busquedas[LIDER].best_params_)
display(pareado)

d = pareado["diferencia"]
print(f'''
Diferencia media     : {d.mean():+.4f}
Desviación de las diferencias : {d.std():.4f}
Pliegues que mejoran : {(d > 0).sum()} de {len(d)}

Si la media es menor que la desviación, la mejora no se distingue del ruido.
Y ojo: los hiperparámetros se eligieron MIRANDO estos mismos pliegues, así que
esta diferencia es un techo, no una estimación.
''')

**Por qué parear importa.** Comparar dos medias obtenidas en particiones
distintas mezcla la mejora con la suerte del reparto. Usando los mismos
pliegues, la diferencia pliegue a pliegue cancela esa fuente de ruido y queda
solo el efecto del cambio.

Con cinco pliegues, sin embargo, la desviación de las diferencias se estima
con cinco números. Que 5 de 5 pliegues mejoren es más informativo que la
magnitud de la media.

In [ ]:
%%time
# Opción A — CV anidada: pliegues externos x combinaciones x pliegues internos.
# Son unos 600 ajustes; entre 30 segundos y 2 minutos según la máquina.
anidada = ml.nested_cv_score(X_train, y_train, name=LIDER)
anidada

In [ ]:
# Las tres cifras, juntas
mejor = busquedas[LIDER].best_score_

print(f'''
Por defecto (Paso 48)               : {por_defecto[LIDER]:.4f}
Mejor de la búsqueda (best_score_)  : {mejor:.4f}    <- OPTIMISTA
CV anidada (procedimiento completo) : {anidada["media"]:.4f}  (+/- {anidada["desviacion"]:.4f})

Sesgo de selección (búsqueda - anidada) : {mejor - anidada["media"]:+.4f}
Ganancia honesta (anidada - por defecto): {anidada["media"] - por_defecto[LIDER]:+.4f}
''')

**Cómo leer las tres cifras.**

La **CV anidada** no mide el mejor punto encontrado: mide el procedimiento
completo *"buscar hiperparámetros y entrenar"*, porque en cada pliegue externo
la búsqueda se rehace desde cero y se evalúa sobre datos que esa búsqueda
nunca vio. Es la cifra que corresponde a lo que ocurriría con datos nuevos.

- **Sesgo de selección** = búsqueda − anidada. Cuánto de `best_score_` era
  haberse quedado con el afortunado entre muchos candidatos.
- **Ganancia honesta** = anidada − por defecto. Lo único que el ajuste aporta
  de verdad. Si sale cerca de cero o negativa, **la configuración por defecto
  era suficiente** y conviene conservarla por simplicidad.

Reportar `best_score_` como si fuera el desempeño esperado es el error más
común de esta parte, y la diferencia entre las dos cifras lo cuantifica.

### El umbral hay que rehacerlo

La sección 7 de `reports/problema_ml.md` dejó registrada una salvedad: el
umbral `0.6086` se derivó sobre `hgb` con parámetros de fábrica. Si el ajuste
cambió el modelo, cambió también la distribución de sus probabilidades.

In [ ]:
# Probabilidades fuera de muestra del modelo ajustado, y su umbral
pipe_ajustado = ml.build_pipeline(ml.get_candidates()[LIDER])
pipe_ajustado.set_params(**busquedas[LIDER].best_params_)
proba_ajustada = ml.oof_proba(pipe_ajustado, X_train, y_train)

u_antes = ml.best_threshold(y_train, proba, criterio="f1")
u_despues = ml.best_threshold(y_train, proba_ajustada, criterio="f1")

display(pd.DataFrame([{"modelo": "por defecto", **u_antes},
                      {"modelo": "ajustado", **u_despues}]))

print(f'''
Brier por defecto : {ml.brier(y_train, proba):.4f}
Brier ajustado    : {ml.brier(y_train, proba_ajustada):.4f}

Si el Brier baja, la regularización también mejoró la calibración, que era el
otro problema que dejó abierto la sección 7.3.
''')

In [ ]:
# ¿La regularización corrigió la forma de S de la calibración?
cal_ajustada = ml.calibration_table(y_train, proba_ajustada, n_bins=8)

comparacion = cal.merge(cal_ajustada, on="tramo", suffixes=("_antes", "_despues"))
comparacion[["tramo", "n_antes", "desvio_antes", "n_despues", "desvio_despues"]]

In [ ]:
# Cierre de la Parte 14: lo que falta para decidir
from sklearn.model_selection import cross_val_score

# 1. Los hiperparámetros completos, sin truncar
print("FINAL_PARAMS candidatos:")
for k, v in busquedas[LIDER].best_params_.items():
    print(f"  {k:32s} {v}")

# 2. Ganancia honesta Y pareada: los MISMOS pliegues externos que usó la CV anidada
externos = ml.make_cv(ml.settings.random_state + 1)
base_ext = cross_val_score(ml.build_pipeline(ml.get_candidates()[LIDER]),
                           X_train, y_train, cv=externos, scoring=ml.PRIMARY_METRIC)
honesto = pd.DataFrame({"pliegue": range(len(base_ext)),
                        "por_defecto": base_ext.round(4),
                        "buscar_y_entrenar": anidada["puntajes"]})
honesto["diferencia"] = (honesto["buscar_y_entrenar"] - honesto["por_defecto"]).round(4)
display(honesto)

dh = honesto["diferencia"]
print(f"\nganancia honesta pareada: {dh.mean():+.4f} (+/- {dh.std():.4f}), "
      f"{(dh > 0).sum()} de {len(dh)} pliegues mejoran\n")

# 3. ¿Cuánto mejoró la calibración, ponderando cada tramo por su número de pasajeros?
for etiqueta, tabla in [("por defecto", cal), ("ajustado", cal_ajustada)]:
    pond = (tabla["desvio"].abs() * tabla["n"]).sum() / tabla["n"].sum()
    print(f"desvío absoluto medio ({etiqueta:11s}), ponderado por n: {pond:.4f}")


# 4. El umbral del modelo ajustado, ahora con el piso de precisión de la sección 2
def fila_u(criterio, u, p_oof):
    p = (p_oof >= u).astype(int)
    return {"criterio": criterio, "umbral": round(float(u), 4),
            "precision": round(precision_score(y_train, p, zero_division=0), 4),
            "recall": round(recall_score(y_train, p, zero_division=0), 4),
            "f1": round(f1_score(y_train, p, zero_division=0), 4),
            "falsos_positivos": int(((p == 1) & (y_train == 0)).sum()),
            "falsos_negativos": int(((p == 0) & (y_train == 1)).sum())}


piso80_aj = ml.threshold_for_precision(y_train, proba_ajustada, min_precision=0.80)
print()
display(pd.DataFrame([
    fila_u("convención (0.5)", 0.5, proba_ajustada),
    fila_u("máximo F1", u_despues["umbral"], proba_ajustada),
    fila_u("precisión >= 0.80", piso80_aj["umbral"], proba_ajustada),
]))
print("¿alcanzó el piso de 0.80?", piso80_aj["alcanzado"])

**Dos cosas que mirar en esa comparación.**

Primero, los **desvíos**: si los de la columna `_despues` son más pequeños y
dejan de apuntar todos en la misma dirección, la regularización corrigió la
forma de S y las probabilidades ya se pueden leer como probabilidades.

Segundo, los **`n`**: un modelo regularizado reparte sus puntajes de forma
menos extrema, así que los tramos de los bordes se vacían y los del medio se
llenan. Eso es esperable y es parte del mismo efecto. Si el tramo superior
queda con muy pocos pasajeros, su desvío deja de ser interpretable, no
empeora.

### Cierre de la Parte 14

Lo que queda decidido, para congelar en la Parte 17:

1. **`FINAL_PARAMS`** — los `best_params_` del modelo elegido, *si* la CV
   anidada muestra una ganancia honesta. Si no, los parámetros por defecto.
2. **`THRESHOLD`** — recalculado sobre el modelo ajustado.
3. **La cifra que se reporta como desempeño esperado** es la de la CV
   anidada, nunca `best_score_`.

El conjunto de prueba sigue sin abrirse. La Parte 16 lo abrirá una sola vez, y
la distancia entre lo que diga y la CV anidada será la medida final de cuánto
optimismo quedaba en todo este procedimiento.

## Parte 15 — Interpretación y ablación de variables

Un modelo predictivo mide qué variables le sirven para **predecir**, no qué
**causa** sobrevivir. Y las explicaciones difieren según el método: por eso
aquí se usan tres y se contrastan.

1. **Coeficientes** del modelo lineal — qué dirección y qué magnitud.
2. **Importancia por permutación** — cuánto se degrada el modelo sin cada
   variable.
3. **Ablación** — qué pasa al quitar grupos enteros de variables.

El Paso 61 resuelve además la duda que el Paso 40 dejó abierta y que la
sección 4 de `reports/problema_ml.md` registró sin cerrar: **¿entran `HasCabin`
y `TCabin` al modelo final?**

### Paso 59. Coeficientes del modelo lineal

In [ ]:
# El modelo lineal ajustado sobre todo el entrenamiento
coef = ml.linear_coefficients(X_train, y_train)
coef

In [ ]:
# Las diez de mayor magnitud, con el signo visible
top = coef.head(10).iloc[::-1]

plt.figure(figsize=(8, 5))
colores = ["tab:green" if c > 0 else "tab:red" for c in top["coeficiente"]]
plt.barh(top["variable"], top["coeficiente"], color=colores, alpha=.85)
plt.axvline(0, color="black", lw=1)
plt.xlabel("coeficiente (positivo empuja a sobrevivir)")
plt.title("Coeficientes del modelo lineal — los 10 de mayor magnitud")
plt.grid(alpha=.3, axis="x")
plt.tight_layout()
plt.show()

**Cómo leer la razón de momios.** Es `exp(coeficiente)`: por cuánto se
multiplican las probabilidades a favor de sobrevivir cuando la variable sube
una unidad. Por encima de 1 empuja hacia sobrevivir, por debajo hacia lo
contrario, y 1 exacto es indiferencia.

**Dos trampas al interpretar estos números.**

Las variables **numéricas están estandarizadas**, así que "una unidad" es una
desviación estándar, no un año de edad ni una libra de tarifa. Un coeficiente
de −0.30 en `num__Age` significa "una desviación estándar más de edad
multiplica los momios por 0.74", no "un año más".

Las **categóricas vienen de one-hot**, así que cada coeficiente compara esa
categoría contra el conjunto de las demás, y no contra una categoría de
referencia fija. `cat__Sex_male` y `cat__Sex_female` son dos columnas
distintas, no una.

Interpreta tres coeficientes por escrito, con la unidad correcta en cada
caso. Es lo que pide el enunciado y es donde se ve si los números se
entendieron o solo se copiaron.

### Paso 60. Importancia por permutación

Se baraja una columna en datos que el modelo **no vio** y se mide cuánto cae
la métrica. Cuanto más cae, más dependía de ella.

Es preferible a la importancia por impurezas de los árboles, que sobrevalora
las variables con muchos valores distintos —`Fare` tiene cientos, `Sex` tiene
dos— por el solo hecho de ofrecer más puntos de corte donde partir.

In [ ]:
# Una partición interna del entrenamiento. X_test NO se toca.
X_tr, X_val, y_tr, y_val = ml.split(X_train, y_train, test_size=0.25)
print(f"ajuste: {len(X_tr)}   permutación: {len(X_val)}")

pipe_perm = ml.build_pipeline(ml.get_candidates()[LIDER])
pipe_perm.set_params(**busquedas[LIDER].best_params_)
pipe_perm.fit(X_tr, y_tr)

perm = ml.permutation_table(pipe_perm, X_val, y_val, n_repeats=30)
perm

In [ ]:
# Media y desviación de la caída, variable por variable
orden = perm.sort_values("caida_media")

plt.figure(figsize=(8, 6))
plt.barh(orden["variable"], orden["caida_media"],
         xerr=orden["desviacion"], color="tab:blue", alpha=.85,
         error_kw={"ecolor": "gray", "capsize": 3})
plt.axvline(0, color="red", ls="--", lw=1)
plt.xlabel("caída de ROC-AUC al barajar la columna")
plt.title(f"Importancia por permutación — {LIDER} ajustado, {len(X_val)} pasajeros")
plt.grid(alpha=.3, axis="x")
plt.tight_layout()
plt.show()

**Qué comparar, y qué debería chirriar.**

Contrasta este orden con dos cosas que ya calculaste: la **matriz de
correlación del Paso 27** y las **tasas por grupo de la Parte 9**.

Lo que se espera que no coincida es `Age`. Su correlación lineal con la
supervivencia era de **−0.08**, prácticamente nula, y sin embargo el EDA
mostró que sí predice: los varones más jóvenes sobrevivieron en torno al 52 %
y los de veintitantos al 14 %. La relación existe pero **no es una línea
recta**, y la correlación no la detecta. Un modelo basado en árboles sí.

Ese contraste es la respuesta a la pregunta del enunciado: si hubieras
descartado variables por su correlación baja, habrías tirado una variable
informativa.

**`HasCabin` y `TCabin` deberían salir en exactamente `0.0000 ± 0.0000`.** No
es casualidad ni un error: el pipeline base no las incluye, así que barajarlas
no puede cambiar ni una predicción. Sirve como comprobación de que la
configuración de variables es la que crees que es — y el Paso 61 es el que
decide si entran.

**Sobre las barras negativas.** Una caída negativa significa que barajar la
columna *mejoró* la métrica. No es que la variable estorbe: es que su aporte
real es cero y el ruido de las 30 repeticiones cayó de ese lado. Mira la
barra de error: si cruza el cero con holgura, la variable no aporta nada
medible.

**Y la advertencia de siempre.** Esto mide de qué depende el modelo, no qué
causó sobrevivir. Dos variables correlacionadas se reparten la importancia y
ambas parecen menos importantes de lo que son: si `Sex` y `TTitulo` son casi
la misma información —la sección 3 de `conclusiones_eda.md` midió una
correlación de −0.80 entre ellas—, barajar una sola deja a la otra cubriendo
el hueco.

### Paso 61. Ablación: ¿cuánto aporta cada grupo de variables?

Reentrenar con y sin un grupo de variables, **sobre los mismos pliegues**, y
medir el cambio. Cuatro configuraciones:

| | Qué cambia | Qué pregunta responde |
|---|---|---|
| **A** | base | la referencia |
| **B** | base + `HasCabin`, `TCabin` | ¿vale la pena el riesgo del artefacto de registro? |
| **C** | base sin `TTitulo` | ¿aporta algo más que duplicar a `Sex`? |
| **D** | base sin `Sex` | ¿qué variables toman el relevo? |

In [ ]:
%%time
# Con los parámetros finales, que son los que se van a desplegar
abl = ml.ablation(X_train, y_train, name=LIDER,
                  params=busquedas[LIDER].best_params_)
abl

In [ ]:
# El cambio frente a la base, contra el ruido de la propia base
base_std = abl.loc[abl["configuracion"] == "A. base", "roc_auc_std"].iloc[0]
orden = abl.iloc[::-1]

plt.figure(figsize=(8, 4))
colores = ["gray" if c.startswith("A.") else
           ("tab:green" if d > 0 else "tab:red")
           for c, d in zip(orden["configuracion"], orden["delta_vs_A"])]
plt.barh(orden["configuracion"], orden["delta_vs_A"], color=colores, alpha=.85)
plt.axvspan(-base_std, base_std, color="gray", alpha=.25,
            label=f"±1 desviación de A ({base_std:.4f})")
plt.axvline(0, color="black", lw=1)
plt.xlabel("Δ ROC-AUC frente a la configuración base")
plt.title("Ablación de variables")
plt.legend(loc="lower right")
plt.grid(alpha=.3, axis="x")
plt.tight_layout()
plt.show()

**La banda gris es el criterio de lectura.** Es la desviación entre pliegues
de la configuración base. Cualquier barra que quede dentro de ella es un
cambio que **no se distingue del ruido**, por mucho que el signo sugiera algo.

**Qué hacer con cada resultado.**

**B — las variables de cabina.** Si la ganancia cae dentro de la banda, se
descartan y el asunto termina ahí. Si la ganancia es clara, la decisión
**no** es automática: la sección 4 de `reports/problema_ml.md` dejó registrado
que la cabina falta en el 18.5 % de primera clase y en el 97.6 % de tercera,
de modo que buena parte de su poder predictivo es `Pclass` disfrazada. Y
queda la sospecha de fondo: que la cabina se registrara con más detalle para
quienes **sobrevivieron para declararla**, lo que la convertiría en un
artefacto del registro y no en un factor de riesgo disponible al embarcar.

> Una variable que mejora la métrica por una razón que no existirá en
> producción es una fuga, no una mejora. La decisión debe justificarse con el
> desempeño **y** con ese argumento, no solo con el número.

**C — `TTitulo`.** Si quitarlo casi no duele, confirma lo que midió el EDA: su
correlación con `Sex` es de −0.80 y aporta poca información nueva.

**D — sin `Sex`.** Aquí el interés no es cuánto cae, sino **qué toma el
relevo**. Vuelve a correr la importancia por permutación sin `Sex` y mira qué
sube: si `TTitulo` ocupa su lugar casi por completo, entonces eliminar el sexo
del modelo por razones legales sería cosmético — la información seguiría
entrando por la puerta del título.

Un detalle que cambia el resultado de D y que conviene mencionar en el
informe: el imputador de edad agrupa por `Sex` y `Pclass`, así que quitar
`Sex` de las variables **no la saca del modelo** —seguiría entrando a través
de la edad imputada—. Por eso `ablation()` reconfigura el imputador para
agrupar solo por `Pclass` cuando `Sex` no está presente.

In [ ]:
# D en detalle: ¿qué variable toma el relevo cuando no está Sex?
num_d, cat_d = ml.ablation_configs()["D. base sin Sex"]

pipe_sin_sexo = ml.build_pipeline(ml.get_candidates()[LIDER], num_d, cat_d)
pipe_sin_sexo.set_params(**busquedas[LIDER].best_params_)
pipe_sin_sexo.set_params(age_imputer__group_cols=("Pclass",))
pipe_sin_sexo.fit(X_tr, y_tr)

perm_sin_sexo = ml.permutation_table(pipe_sin_sexo, X_val[num_d + cat_d],
                                     y_val, n_repeats=30)

comparacion = (perm.set_index("variable")[["caida_media"]]
               .rename(columns={"caida_media": "con_Sex"})
               .join(perm_sin_sexo.set_index("variable")[["caida_media"]]
                     .rename(columns={"caida_media": "sin_Sex"}), how="outer"))
comparacion["cambio"] = (comparacion["sin_Sex"] - comparacion["con_Sex"]).round(4)
comparacion.sort_values("cambio", ascending=False)

### Cierre de la Parte 15

Lo que queda decidido:

1. **`USE_CABIN_FEATURES`** — la última constante que faltaba para congelar el
   módulo en el Paso 64. Se decide con la fila B y con el argumento del
   artefacto de registro, no solo con el número.
2. **La explicación del modelo** — coeficientes e importancia por permutación
   entran en la ficha del modelo (`reports/model_card.md`, Paso 69).
3. **Qué tan sustituible es `Sex`** — la última tabla responde si quitarlo
   sería una medida real o cosmética, y eso pertenece a la sección 7.6 de
   `reports/problema_ml.md`, donde se listan los usos no recomendados.

El conjunto de prueba sigue sin abrirse. Ya solo queda la Parte 16 para
abrirlo, una sola vez.

## Parte 16 — Evaluación final sobre el conjunto de prueba

Todo lo anterior se decidió con validación cruzada sobre el entrenamiento: el
modelo, los hiperparámetros, el umbral y las variables. El conjunto de prueba
no ha participado en ninguna de esas decisiones.

Ahora se abre. **Una vez.**

> **La regla que hace que esta cifra signifique algo.** Si el resultado
> decepciona, **no** se vuelve atrás a arreglarlo. Cualquier ajuste hecho
> después de mirar la prueba la convierte en un segundo conjunto de
> validación, y entonces deja de estimar el desempeño con datos nuevos. Si
> sale mal, se documenta tal cual y se explica por qué.

### Paso 62. Evaluar en prueba con incertidumbre

In [ ]:
# El modelo con las decisiones congeladas, entrenado SOLO con X_train
pipe_final = ml.train_final(X_train, y_train)
proba_test = pipe_final.predict_proba(X_test)[:, 1]

prueba = ml.evaluate_on_test(pipe_final, X_test, y_test, ml.THRESHOLD)
pd.Series(prueba).to_frame("valor")

In [ ]:
# Un número solo no dice cuánto puede moverse: intervalo por bootstrap
ics = [ml.bootstrap_ci(y_test, proba_test, m, ml.THRESHOLD)
       for m in ["roc_auc", "accuracy", "f1"]]
tabla_ic = pd.DataFrame(ics)
tabla_ic

In [ ]:
# Las tres estimaciones con su intervalo del 95 %
plt.figure(figsize=(8, 3.2))
for i, r in enumerate(ics):
    plt.errorbar(r["estimacion"], i,
                 xerr=[[r["estimacion"] - r["ic_bajo"]],
                       [r["ic_alto"] - r["estimacion"]]],
                 fmt="o", capsize=6, color="tab:blue", markersize=8)
    plt.text(r["ic_alto"] + 0.01, i, f"ancho {r['amplitud']:.3f}",
             va="center", fontsize=9, color="gray")
plt.axvline(0.5, color="red", ls="--", lw=1, label="azar (ROC-AUC)")
plt.yticks(range(len(ics)), [r["metrica"] for r in ics])
plt.xlim(0.4, 1.05)
plt.xlabel("valor en el conjunto de prueba")
plt.title(f"Evaluación final — {len(y_test)} pasajeros, IC 95 % por bootstrap")
plt.legend(loc="lower left")
plt.grid(alpha=.3, axis="x")
plt.tight_layout()
plt.show()

**El ancho del intervalo es el resultado más importante de esta parte.**

Con unos 180 pasajeros de prueba, el intervalo del ROC-AUC abarca del orden de
10 puntos. Eso significa que **esta muestra no puede distinguir dos modelos
que difieran en menos de eso**.

Mira hacia atrás con ese dato en la mano:

- La Parte 12 separaba a `hgb` de `forest` por 0.021 puntos de ROC-AUC.
- La Parte 14 discutía una ganancia de 0.0058 que resultó ser ruido.
- La Parte 15 medía ablaciones de 0.002 a 0.010.

**Todas esas diferencias caben varias veces dentro de este intervalo.** No
significa que el trabajo fuera inútil —sirvió para descartar lo que claramente
no funcionaba y para entender el modelo—, pero sí que las decisiones finas
entre configuraciones equivalentes no estaban sostenidas por los datos. Decirlo
es más valioso que presentar un ranking con cuatro decimales.

### Paso 63. Comparar, contrastar y documentar

In [ ]:
# Todo el recorrido en una tabla: de la vara al resultado final
por_modelo = resultados.set_index("modelo")

comparacion_final = pd.DataFrame([
    {"modelo": "dummy (clase mayoritaria)", "medido_en": "CV entrenamiento",
     "roc_auc": por_modelo.loc["dummy", "roc_auc_mean"],
     "exactitud": por_modelo.loc["dummy", "accuracy_mean"]},
    {"modelo": "regla de género", "medido_en": "entrenamiento",
     "roc_auc": None, "exactitud": round(float(VARA), 4)},
    {"modelo": f"{LIDER} por defecto", "medido_en": "CV entrenamiento",
     "roc_auc": por_modelo.loc[LIDER, "roc_auc_mean"],
     "exactitud": por_modelo.loc[LIDER, "accuracy_mean"]},
    {"modelo": f"{LIDER} ajustado", "medido_en": "CV anidada",
     "roc_auc": anidada["media"], "exactitud": None},
    {"modelo": f"{LIDER} ajustado", "medido_en": "PRUEBA (una vez)",
     "roc_auc": prueba["roc_auc"], "exactitud": prueba["accuracy"]},
])
comparacion_final

In [ ]:
# La vara, medida sobre el MISMO conjunto de prueba
from sklearn.metrics import accuracy_score, roc_auc_score

pred_genero = ml.gender_rule_predict(X_test)

print(f"""
Sobre los {len(y_test)} pasajeros de prueba:

  regla de género   exactitud = {accuracy_score(y_test, pred_genero):.4f}   F1 = {f1_score(y_test, pred_genero):.4f}
  {ml.FINAL_ESTIMATOR} ajustado    exactitud = {prueba['accuracy']:.4f}   F1 = {prueba['f1']:.4f}

  margen del modelo sobre la regla: {prueba['accuracy'] - accuracy_score(y_test, pred_genero):+.4f}
  (el criterio de la sección 3 pedía al menos +0.02)
""")

In [ ]:
# ¿La prueba es coherente con lo que anticipó la validación cruzada?
esperado = anidada["media"]
observado = prueba["roc_auc"]
ic = ics[0]

print(f'''
Anticipado por la CV anidada : {esperado:.4f} (+/- {anidada["desviacion"]:.4f})
Observado en prueba          : {observado:.4f}  IC 95 % [{ic["ic_bajo"]:.4f}, {ic["ic_alto"]:.4f}]
Diferencia                   : {observado - esperado:+.4f}

¿La anticipación cae dentro del intervalo de la prueba?  {ic["ic_bajo"] <= esperado <= ic["ic_alto"]}
''')

**Cómo interpretar la coherencia.**

Si la estimación de la CV anidada cae **dentro** del intervalo de la prueba, el
procedimiento de evaluación funcionó: lo que se anticipó es compatible con lo
que ocurrió, y no hubo fuga ni sobreajuste a las decisiones.

Si cae **fuera**, hay algo que explicar, y las explicaciones posibles no son
equivalentes:

| Qué se observa | Explicación más probable |
|---|---|
| La prueba sale **peor** | Sobreajuste a las decisiones tomadas con el entrenamiento, o una partición de prueba desfavorable. |
| La prueba sale **mejor** | Normalmente azar de la partición. Con 180 pasajeros, varios puntos de diferencia no necesitan más explicación. |
| La diferencia es **grande en cualquier dirección** | Revisar que la partición siga siendo la misma (`random_state`) y que nada del conjunto de prueba se haya colado en el entrenamiento. |

En cualquiera de los casos, **la cifra que se reporta es la de la prueba**, con
su intervalo, y la discrepancia se documenta. No se corrige.

## Parte 17 — Entrenamiento final y guardado del modelo

Terminada la investigación, se **congelan las decisiones** y se entrena el
modelo definitivo con **todos** los datos etiquetados, prueba incluida.

No es trampa: el procedimiento ya está validado y su desempeño esperado es el
que acaba de medir la Parte 16. Más datos suelen dar un modelo algo mejor, y
desaprovechar 179 pasajeros una vez terminada la investigación no tendría
sentido.

### Paso 64. Las decisiones, congeladas en el código

No viven en la memoria de quien hizo el experimento ni solo en el informe:
viven en `flows/05_ml_models.py`, bajo control de versiones.

In [ ]:
# Las seis constantes y de dónde salió cada una
num_f, cat_f = ml.final_features()

print(f'''
MODEL_NAME          = {ml.MODEL_NAME!r}
FINAL_ESTIMATOR     = {ml.FINAL_ESTIMATOR!r}          <- Paso 50, sección 6
USE_CABIN_FEATURES  = {ml.USE_CABIN_FEATURES}              <- Paso 61, sección 9.4
THRESHOLD           = {ml.THRESHOLD}              <- Paso 53, secciones 7.2 y 8.6

FINAL_PARAMS                            <- Paso 57, sección 8.6''')
for k, v in ml.FINAL_PARAMS.items():
    print(f"  {k:32s} {v}")
print(f"\nVariables numéricas  ({len(num_f)}): {num_f}")
print(f"Variables categóricas ({len(cat_f)}): {cat_f}")

### Pasos 65, 66 y 68. Entrenar, guardar y orquestar

`ml.run()` hace el recorrido completo: lee Silver, evalúa una vez sobre la
prueba, entrena el modelo definitivo con todos los datos etiquetados y lo
guarda junto con sus metadatos.

Es la misma función que ejecuta `make pipeline` como última etapa.

In [ ]:
%%time
ruta_modelo = ml.run()
print("\nmodelo:", ruta_modelo.name)
print("metadatos:", ruta_modelo.with_suffix(".metadata.json").name)

In [ ]:
# Qué quedó guardado junto al modelo
import json

meta = json.loads(ruta_modelo.with_suffix(".metadata.json").read_text(encoding="utf-8"))

print("--- procedencia ---")
for k in ["name", "estimator", "target", "threshold", "random_state", "n_train"]:
    print(f"  {k:16s} {meta[k]}")
print(f"  {'data_sha256':16s} {meta['data_sha256'][:16]}…")
print(f"  {'model_sha256':16s} {meta['model_sha256'][:16]}…")

print("\n--- entorno ---")
for k in ["saved_at", "python_version", "sklearn_version", "pandas_version"]:
    print(f"  {k:16s} {meta[k]}")

print("\n--- desempeño reportado (conjunto de prueba) ---")
for k in ["roc_auc", "accuracy", "precision", "recall", "f1", "brier", "n_test"]:
    print(f"  {k:16s} {meta['metrics'][k]}")
ic = meta["metrics"]["roc_auc_ic95"]
print(f"  {'roc_auc IC 95%':16s} [{ic['ic_bajo']}, {ic['ic_alto']}]")

**Por qué los metadatos no son burocracia.** Dentro de seis meses, ante un
modelo en producción que empieza a fallar, estas son las preguntas que alguien
tendrá que responder:

| Pregunta | Qué la responde |
|---|---|
| ¿Con qué datos exactos se entrenó? | `data_sha256` — la huella del Parquet |
| ¿Es este el archivo que se guardó? | `model_sha256` |
| ¿Qué umbral usa? | `threshold`, que viaja con el modelo |
| ¿Se puede reproducir? | `random_state` + las versiones |
| ¿Qué desempeño se prometió? | `metrics`, con su intervalo |

> **Seguridad.** Un `.joblib` es un `pickle`: puede ejecutar código al
> cargarse. Nunca cargues un modelo de una fuente que no controles. Y un
> modelo guardado con una versión de scikit-learn puede no abrirse con otra;
> por eso `load_model()` avisa cuando la versión cambió.

### Paso 67. Cargar y predecir

In [ ]:
# Ida y vuelta completa: cargar desde disco y predecir
pipe_cargado, meta_cargada = ml.load_model(ruta_modelo)

muestra = ml.predict(pipe_cargado, X_test.head(8))
muestra.join(X_test.head(8)[["Sex", "Pclass", "Age", "TTitulo"]]).assign(
    real=y_test.head(8).to_numpy()
)

### Paso 69. Prueba de humo con pasajeros sin etiqueta

Los 418 pasajeros de Bronze que **no tienen desenlace conocido**. No hay
métrica que calcular: el objetivo es verificar que el flujo completo —carga,
preprocesamiento, predicción— corre de punta a punta con datos que el modelo
nunca vio.

In [ ]:
# Las mismas transformaciones deterministas de Silver, sobre los no etiquetados
from flows.common.paths import BRONZE, BRONZE_UNLABELED_NAME

de = importlib.import_module("flows.01_data_engineering")
nuevos = pd.read_csv(BRONZE / BRONZE_UNLABELED_NAME)
print(f"pasajeros sin etiqueta: {len(nuevos)}")

# build_silver exige Survived por su contrato de entrada (validate_bronze) y la
# usa en add_survivor_label. Se añade un marcador que no entra al modelo:
# CANDIDATE_COLUMNS no incluye ni Survived ni Survivor.
silver_nuevos = de.build_silver(nuevos.assign(Survived=0))

X_nuevos = silver_nuevos[ml.CANDIDATE_COLUMNS].copy()
X_nuevos["Age"] = silver_nuevos["AgeOriginal"]      # igual que load_dataset()

salida = ml.predict(pipe_cargado, X_nuevos)
salida.head()

In [ ]:
# Las tres comprobaciones que pide el paso
print("1. ¿Hay NaN en las predicciones?", int(salida.isna().sum().sum()))

print("\n2. ¿La distribución de probabilidades es razonable?")
print(salida["proba_survive"].describe().round(4).to_string())
print(f"   tasa predicha de supervivencia: {salida['pred'].mean():.4f}")
print(f"   tasa observada en entrenamiento: {y.mean():.4f}")

print("\n3. ¿Alguna categoría que el modelo no conoce?")
conocidas = ml.load_dataset()[0]
hay_nuevas = False
for col in cat_f:
    nuevas = set(X_nuevos[col].dropna().unique()) - set(conocidas[col].dropna().unique())
    if nuevas:
        hay_nuevas = True
        print(f"   {col}: {nuevas}")
if not hay_nuevas:
    print("   ninguna")

In [ ]:
# Las dos distribuciones de probabilidad, lado a lado
plt.figure(figsize=(9, 4.5))
plt.hist(proba_test, bins=25, alpha=.6, density=True,
         label=f"prueba etiquetada (n={len(proba_test)})")
plt.hist(salida["proba_survive"], bins=25, alpha=.6, density=True,
         label=f"sin etiqueta (n={len(salida)})")
plt.axvline(ml.THRESHOLD, color="red", ls="--", lw=1.5,
            label=f"THRESHOLD = {ml.THRESHOLD}")
plt.xlabel("probabilidad predicha de sobrevivir")
plt.ylabel("densidad")
plt.title("¿El modelo se comporta igual con datos nuevos?")
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

**Qué debería verse y qué sería una alarma.**

Las dos distribuciones deberían tener una forma **parecida**, aunque no
idéntica: los 418 sin etiqueta son otra muestra de los mismos pasajeros y su
composición por clase y sexo no es exactamente igual.

Serían señales de problema:

- Una distribución **concentrada en el centro** en los nuevos y no en los de
  prueba: el modelo no está discriminando, probablemente porque alguna
  variable llegó vacía.
- Una **tasa predicha muy distinta** de la observada en entrenamiento sin que
  la composición lo explique.
- **Categorías nuevas** en algún campo. El `OneHotEncoder` está configurado con
  `handle_unknown="infrequent_if_exist"`, así que no falla — pero conviene
  saber que ocurrió.

No hay métrica porque no hay respuesta: la evaluación de estos 418 pasajeros
corresponde al Taller #3.

### Cierre: verificación de extremo a extremo

El taller termina cuando alguien puede clonar el repositorio y reproducir todo
sin intervención manual:

```bash
make install
make test         # todas las pruebas en verde
make lint         # sin errores
make pipeline     # ingesta -> silver -> gold -> modelo guardado
ls data/bronze data/silver data/gold models
```

Falta un último entregable: **`reports/model_card.md`**, la ficha del modelo.
Una página con el propósito y el uso previsto, los datos de entrenamiento con
su huella SHA-256, las variables usadas y excluidas, las métricas de validación
y de prueba **con su incertidumbre**, el desempeño por subgrupos del Paso 54,
las limitaciones del Paso 63 y cómo reproducir todo.

Es el documento que lee quien hereda el modelo, y el que convierte un archivo
`.joblib` en algo auditable.